In [1]:
import os
import pandas as pd

folder_path = r"C:\Users\konak\Downloads\Maven+Fuzzy+Factory"
output_file = "Maven_Fuzzy_Factory_Merged.xlsx"

def merge_my_folder_with_splitting(path, filename):
    if not os.path.exists(path):
        print(f"❌ Error: The folder path '{path}' does not exist.")
        return

    valid_extensions = ('.csv', '.xlsx', '.xls')
    files = [f for f in os.listdir(path) if f.lower().endswith(valid_extensions)]
    
    if not files:
        print(f"⚠️ No compatible CSV or Excel files found inside: {path}")
        return

    output_path = os.path.join(os.path.dirname(path), filename)
    print(f"🔄 Found {len(files)} files. Merging (and splitting large files if needed)...")
    
    EXCEL_ROW_LIMIT = 1048500  # Keeping it slightly under 1,048,576 for safety
    
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        for file in files:
            file_path = os.path.join(path, file)
            base_name = os.path.splitext(file)[0]
            
            try:
                # Read data based on extension
                if file.lower().endswith('.csv'):
                    df = pd.read_csv(file_path)
                else:
                    df = pd.read_excel(file_path)
                
                total_rows = len(df)
                
                # Check if the file fits in a single Excel sheet
                if total_rows <= EXCEL_ROW_LIMIT:
                    sheet_name = base_name[:30]
                    df.to_excel(writer, sheet_name=sheet_name, index=False)
                    print(f"   ✅ Created sheet: [{sheet_name}] ({total_rows} rows)")
                else:
                    # File is too large -> Split it into chunks
                    print(f"   ⚠️ '{file}' has {total_rows} rows (Exceeds Excel limit). Splitting...")
                    
                    chunk_num = 1
                    for start_idx in range(0, total_rows, EXCEL_ROW_LIMIT):
                        df_chunk = df.iloc[start_idx : start_idx + EXCEL_ROW_LIMIT]
                        
                        # Generate a safe sheet name like 'website_pageviews_1'
                        suffix = f"_{chunk_num}"
                        sheet_name = f"{base_name[:28]}{suffix}"
                        
                        df_chunk.to_excel(writer, sheet_name=sheet_name, index=False)
                        print(f"      👉 Created split sheet: [{sheet_name}] ({len(df_chunk)} rows)")
                        chunk_num += 1
                        
            except Exception as e:
                print(f"   ❌ Skipped '{file}' due to an error: {e}")
                
    print(f"\n🎉 Process Complete! Your master file is saved at:\n👉 {output_path}\n")

# Run the updated function
merge_my_folder_with_splitting(folder_path, output_file)


🔄 Found 7 files. Merging (and splitting large files if needed)...
   ✅ Created sheet: [maven_fuzzy_factory_data_dicti] (36 rows)
   ✅ Created sheet: [orders] (32313 rows)
   ✅ Created sheet: [order_items] (40025 rows)
   ✅ Created sheet: [order_item_refunds] (1731 rows)
   ✅ Created sheet: [products] (4 rows)
   ⚠️ 'website_pageviews.csv' has 1188124 rows (Exceeds Excel limit). Splitting...
      👉 Created split sheet: [website_pageviews_1] (1048500 rows)
      👉 Created split sheet: [website_pageviews_2] (139624 rows)
   ✅ Created sheet: [website_sessions] (472871 rows)

🎉 Process Complete! Your master file is saved at:
👉 C:\Users\konak\Downloads\Maven_Fuzzy_Factory_Merged.xlsx



In [ ]:
pip install pandas openpyxl ipywidgets


In [ ]:
pip install pandas